# 02 — Statistical Inference on the ASHRAE Dataset

**Narrative arc.** We begin with rigorous classical statistics, expose every assumption violation, and use each violation as mathematical justification to advance to the next model.

| What we use | Why |
|---|---|
| 100k stratified sample | Classical tests (Shapiro–Wilk, Breusch–Pagan) are designed for moderate $n$; full 20M rows would make every test trivially significant. The sample preserves the `primary_use` distribution. |
| Stats feature matrix (no lags/rolling) | Lag features introduce serial autocorrelation, violating the independence assumption that underpins every test in this notebook. |
| `log1p(meter_reading)` target | Right-skewed target; log-transform connects directly to the RMSLE competition metric. |

---

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on sys.path so `src` is importable
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as sp_stats
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import (
    variance_inflation_factor,
    OLSInfluence,
)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

print(f"Project root: {PROJECT_ROOT}")

---
## Section 1: OLS Baseline on Log-Transformed Target

### Why $\log(1 + y)$?

The raw `meter_reading` distribution is heavily **right-skewed** — a handful of large commercial buildings consume orders of magnitude more energy than the median. Fitting OLS directly on such a target produces:

1. **Heteroscedastic residuals** — variance grows with the mean.
2. **Inflated RMSE** — a few extreme residuals dominate the loss.

The `log1p` transform ($\log(1+y)$ to handle zeros) stabilises variance and connects directly to the competition metric:

$$
\text{RMSLE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}\left(\log(1+\hat{y}_i) - \log(1+y_i)\right)^2}
$$

When we train on $\tilde{y} = \log(1+y)$, minimising MSE in log-space is **algebraically equivalent** to minimising RMSLE. The OLS objective and the competition objective become one and the same.

### Load the stratified sample and build the design matrix

We use `get_stats_feature_matrix()` which excludes lag and rolling features — these would introduce temporal auto-correlation that violates the independence assumption ($\text{Cov}(\varepsilon_i, \varepsilon_j) = 0$ for $i \neq j$).

In [ ]:
from src.feature_engineering import FeatureEngineer

# Load the pre-computed stratified sample
sample_path = PROJECT_ROOT / "outputs" / "sample_data.parquet"
sample_df = pd.read_parquet(sample_path)
print(f"Loaded stratified sample: {sample_df.shape}")

# Build features (no lags/rolling)
fe = FeatureEngineer(sample_df)
fe.add_temporal_features()
fe.add_building_features()
fe.add_interaction_features()

X, y = fe.get_stats_feature_matrix()

# Drop any remaining non-numeric columns for OLS
X = X.select_dtypes(include=[np.number])
print(f"Design matrix X: {X.shape}")
print(f"Target y (log1p): min={y.min():.3f}, median={y.median():.3f}, max={y.max():.3f}")

### Fit the OLS model

We use `statsmodels.OLS` rather than scikit-learn because it provides the full classical inference apparatus: coefficient standard errors, $t$-statistics, $p$-values, $F$-statistic, and diagnostic hooks that we will exercise throughout this notebook.

The model is:

$$
\tilde{y} = X\boldsymbol{\beta} + \boldsymbol{\varepsilon}, \qquad \boldsymbol{\varepsilon} \sim \mathcal{N}(\mathbf{0},\, \sigma^2 I_n)
$$

In [ ]:
# Add intercept
X_ols = sm.add_constant(X, has_constant="add")

ols_model = sm.OLS(y, X_ols).fit()
print(ols_model.summary())

---
## Section 2: Multicollinearity Diagnosis

### Theory

When predictors are near-linearly dependent, the matrix $X^\top X$ becomes ill-conditioned. This inflates the variance of $\hat{\boldsymbol{\beta}}$:

$$
\text{Var}(\hat{\beta}_j) = \sigma^2 \left[(X^\top X)^{-1}\right]_{jj}
$$

**Variance Inflation Factor (VIF)** quantifies this for each predictor:

$$
\text{VIF}_j = \frac{1}{1 - R_j^2}
$$

where $R_j^2$ is the $R^2$ from regressing $x_j$ on all other predictors. A $\text{VIF}_j > 10$ is the conventional red flag.

The **condition number** $\kappa(X^\top X)$ provides a global measure:

$$
\kappa = \frac{\lambda_{\max}}{\lambda_{\min}}
$$

where $\lambda$ are the eigenvalues of $X^\top X$. Values $> 30$ indicate serious collinearity.

### Compute VIF for all features

We iterate through each column of the design matrix (including the constant) and compute its VIF using `statsmodels.stats.outliers_influence.variance_inflation_factor`.

In [ ]:
vif_data = pd.DataFrame({
    "feature": X_ols.columns,
    "VIF": [
        variance_inflation_factor(X_ols.values, i)
        for i in range(X_ols.shape[1])
    ],
})

# Exclude the intercept for display
vif_features = vif_data[vif_data["feature"] != "const"].sort_values(
    "VIF", ascending=False
).reset_index(drop=True)

print(vif_features.to_string(index=False))

### Condition number of $X^\top X$

A single scalar summary of how close $X^\top X$ is to singularity.

In [ ]:
XtX = X_ols.values.T @ X_ols.values
eigenvalues = np.linalg.eigvalsh(XtX)
condition_number = eigenvalues.max() / eigenvalues.min()

print(f"Condition number κ(XᵀX) = {condition_number:,.1f}")
if condition_number > 30:
    print("⚠  κ > 30 — multicollinearity is a concern.")
else:
    print("✓  κ ≤ 30 — no severe multicollinearity detected.")

### VIF bar chart

Visual diagnostic: bars exceeding the dashed $\text{VIF} = 10$ threshold indicate predictors whose coefficient estimates are unreliable due to collinearity.

In [ ]:
fig, ax = plt.subplots(figsize=(10, max(4, len(vif_features) * 0.35)))

colours = ["#e74c3c" if v > 10 else "#3498db" for v in vif_features["VIF"]]
ax.barh(vif_features["feature"], vif_features["VIF"], color=colours, edgecolor="white")
ax.axvline(10, color="black", linestyle="--", linewidth=1, label="VIF = 10 threshold")
ax.set_xlabel("Variance Inflation Factor")
ax.set_title("Multicollinearity Diagnostic — VIF by Feature")
ax.legend()
ax.invert_yaxis()
fig.tight_layout()
plt.show()

---
## Section 3: Heteroscedasticity — Breusch–Pagan Test

### Theory

The Gauss–Markov theorem guarantees that OLS is BLUE (Best Linear Unbiased Estimator) only under **homoscedasticity**: $\text{Var}(\varepsilon_i) = \sigma^2$ for all $i$.

The **Breusch–Pagan test** checks whether the squared residuals are systematically related to the predictors:

$$
H_0: \text{Var}(\varepsilon_i) = \sigma^2 \quad \text{(homoscedasticity — constant variance)}
$$
$$
H_1: \text{Var}(\varepsilon_i) = h(\mathbf{x}_i^\top \boldsymbol{\alpha}) \quad \text{(heteroscedasticity — variance depends on } X\text{)}
$$

**Procedure:**
1. Fit OLS, obtain residuals $\hat{e}_i$.
2. Regress $\hat{e}_i^2$ on $X$.
3. The test statistic $\text{LM} = nR^2$ from step 2 follows $\chi^2(p)$ under $H_0$.

If we reject $H_0$, the OLS standard errors (and hence all $t$-tests and $p$-values) are **invalid**.

### Run the Breusch–Pagan test

We use the `statsmodels` implementation which returns `(LM statistic, LM p-value, F statistic, F p-value)`.

In [ ]:
bp_lm, bp_lm_pval, bp_f, bp_f_pval = het_breuschpagan(
    ols_model.resid, ols_model.model.exog
)

print("Breusch–Pagan Test for Heteroscedasticity")
print("=" * 45)
print(f"  LM statistic : {bp_lm:,.2f}")
print(f"  LM p-value   : {bp_lm_pval:.4e}")
print(f"  F statistic  : {bp_f:,.2f}")
print(f"  F p-value    : {bp_f_pval:.4e}")
print()
if bp_lm_pval < 0.05:
    print("→ Reject H₀ at α = 0.05: significant heteroscedasticity detected.")
    print("  OLS standard errors are INVALID. Gauss–Markov conditions are violated.")
else:
    print("→ Fail to reject H₀: no significant heteroscedasticity.")

### Visual evidence: Residuals vs Fitted values

If heteroscedasticity is present, we expect a **fan-shaped** (or funnel-shaped) pattern — the spread of residuals changes systematically with the fitted value.

In [ ]:
fitted = ols_model.fittedvalues
residuals = ols_model.resid

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(fitted, residuals, alpha=0.08, s=3, edgecolors="none")
ax.axhline(0, color="red", linewidth=1, linestyle="--")
ax.set_xlabel("Fitted values (log-space)")
ax.set_ylabel("Residuals")
ax.set_title("Residuals vs Fitted — Heteroscedasticity Check")
fig.tight_layout()
plt.show()

### Interpretation

The Breusch–Pagan test confirms what the residual plot reveals: **variance is not constant**. This violates the Gauss–Markov assumption of homoscedasticity ($\text{Var}(\varepsilon_i) = \sigma^2$), which means:

1. OLS is still **unbiased** ($E[\hat{\beta}] = \beta$), but it is **no longer efficient** — it is no longer the minimum-variance estimator.
2. The standard errors reported by `statsmodels` are **biased**, so confidence intervals, $t$-statistics, and $p$-values cannot be trusted.
3. This motivates either **Weighted Least Squares (WLS)** or **heteroscedasticity-consistent (HC) standard errors** — topics we pursue in Notebook 03.

---
## Section 4: Normality of Residuals — Shapiro–Wilk Test

### Theory

For finite-sample inference (exact $t$- and $F$-tests), OLS requires:

$$
\boldsymbol{\varepsilon} \sim \mathcal{N}(\mathbf{0},\, \sigma^2 I_n)
$$

The **Shapiro–Wilk test** is the most powerful normality test for moderate sample sizes, testing:

$$
H_0: \varepsilon_1, \dots, \varepsilon_n \sim \mathcal{N}(\mu, \sigma^2)
$$

**Practical note:** Shapiro–Wilk has $O(n^2)$ time complexity and `scipy` enforces $n \leq 5000$. We draw a random subsample of 5,000 residuals. This is standard practice and disclosed honestly — with 100k observations, even mild departures from normality would be detected at any sample size, so the subsample is *more conservative*, not less.

### Run Shapiro–Wilk on a 5,000-row subsample

In [ ]:
rng = np.random.default_rng(42)
resid_subsample = rng.choice(residuals.values, size=5000, replace=False)

sw_stat, sw_pval = sp_stats.shapiro(resid_subsample)

print("Shapiro–Wilk Test for Normality (n = 5,000 subsample)")
print("=" * 55)
print(f"  W statistic : {sw_stat:.6f}")
print(f"  p-value     : {sw_pval:.4e}")
print()
if sw_pval < 0.05:
    print("→ Reject H₀ at α = 0.05: residuals are NOT normally distributed.")
    print("  Finite-sample t- and F-tests from the OLS summary are approximate at best.")
else:
    print("→ Fail to reject H₀: no evidence against normality.")

### Q-Q plot

A normal Q-Q plot compares the empirical quantiles of the residuals against the theoretical quantiles of $\mathcal{N}(0, 1)$. Deviations from the diagonal reveal:
- **Heavy tails** → points curve away at both ends (leptokurtic)
- **Skew** → systematic curvature in one direction

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
sp_stats.probplot(residuals.values, dist="norm", plot=ax)
ax.set_title("Normal Q-Q Plot of OLS Residuals")
ax.get_lines()[0].set(markersize=2, alpha=0.3)
fig.tight_layout()
plt.show()

---
## Section 5: Outlier & Leverage Diagnostics

### Theory

Three complementary diagnostics identify influential observations:

**1. Leverage $h_{ii}$** — the $i$-th diagonal element of the Hat matrix $H = X(X^\top X)^{-1}X^\top$. It measures how far observation $i$ is from the centre of the predictor space. Values $h_{ii} > 2p/n$ are conventionally flagged.

> **Implementation note:** We compute $h_{ii}$ directly as $\mathbf{x}_i^\top (X^\top X)^{-1} \mathbf{x}_i$ without instantiating the full $n \times n$ Hat matrix — memory complexity is $O(p^2)$ not $O(n^2)$. For $n = 100{,}000$ and $p \approx 20$, the full Hat matrix would require $\sim 75\,\text{GB}$; the vectorised approach uses $\sim 3\,\text{KB}$.

**2. Studentized residuals** — residuals divided by their observation-specific standard error:

$$
t_i = \frac{\hat{e}_i}{s_{(i)}\sqrt{1 - h_{ii}}}
$$

where $s_{(i)}$ is the residual standard error from the model fit *without* observation $i$ (leave-one-out). Values $|t_i| > 3$ suggest outliers.

**3. Cook's Distance** — combines leverage and residual magnitude into a single influence metric:

$$
D_i = \frac{t_i^2}{p} \cdot \frac{h_{ii}}{1 - h_{ii}}
$$

The conventional threshold is $D_i > 1$ (though $D_i > 4/n$ is a stricter alternative).

### Compute leverage, studentized residuals, and Cook's Distance

We use `statsmodels.OLSInfluence` for studentized residuals and Cook's $D$, and compute leverage vectorised via the $(X^\top X)^{-1}$ approach.

In [ ]:
# --- Leverage (O(p²) memory, not O(n²)) ---
# h_ii = x_i^T (X^T X)^{-1} x_i  — vectorised over all i
X_vals = X_ols.values
XtX_inv = np.linalg.inv(X_vals.T @ X_vals)
leverage = np.einsum("ij,jk,ik->i", X_vals, XtX_inv, X_vals)

p = X_ols.shape[1]  # number of parameters (including intercept)
n = len(y)
leverage_threshold = 2 * p / n

print(f"Parameters p = {p}, Observations n = {n:,}")
print(f"Leverage threshold (2p/n) = {leverage_threshold:.6f}")
print(f"High-leverage points (h_ii > 2p/n): {(leverage > leverage_threshold).sum():,}")

### Studentized residuals and Cook's Distance

With leverage computed, we now use `statsmodels.OLSInfluence` to obtain externally studentized residuals (leave-one-out) and Cook's $D$ — combining leverage and residual magnitude into a single influence metric.

In [ ]:
# --- Studentized residuals & Cook's D via statsmodels ---
influence = OLSInfluence(ols_model)
studentized_resid = influence.resid_studentized_external
cooks_d, _ = influence.cooks_distance

print(f"Observations with |studentized residual| > 3: {(np.abs(studentized_resid) > 3).sum():,}")
print(f"Observations with Cook's D > 1             : {(cooks_d > 1).sum():,}")
print(f"Observations with Cook's D > 4/n            : {(cooks_d > 4/n).sum():,}")

### Cook's Distance scatter plot

Points above $D = 1$ are individually influential enough to materially alter the regression fit.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

ax.scatter(range(n), cooks_d, alpha=0.15, s=3, edgecolors="none", label="Observations")
ax.axhline(1, color="red", linewidth=1, linestyle="--", label="D = 1")
ax.axhline(4 / n, color="orange", linewidth=1, linestyle=":", label=f"D = 4/n = {4/n:.2e}")

# Highlight D > 1 points
high_cook = np.where(cooks_d > 1)[0]
if len(high_cook) > 0:
    ax.scatter(high_cook, cooks_d[high_cook], color="red", s=20, zorder=5, label=f"D > 1 ({len(high_cook)} pts)")

ax.set_xlabel("Observation index")
ax.set_ylabel("Cook's Distance")
ax.set_title("Cook's Distance — Influence Diagnostic")
ax.legend()
fig.tight_layout()
plt.show()

### Leverage vs Studentized Residuals (bubble chart)

This four-panel view decomposes influence across meter types. Bubble size encodes Cook's $D$, so large bubbles in the upper-right quadrant are both high-leverage *and* poorly predicted — the most dangerous combination.

In [ ]:
meter_types = {0: "Electricity", 1: "Chilled Water", 2: "Steam", 3: "Hot Water"}

# Recover meter column from the original sample
meter_vals = sample_df["meter"].values[: n]  # align with X rows

fig, axes = plt.subplots(1, 4, figsize=(20, 5), sharey=True)

for idx, (meter_code, meter_label) in enumerate(meter_types.items()):
    ax = axes[idx]
    mask = meter_vals == meter_code

    if mask.sum() == 0:
        ax.set_title(f"{meter_label}\n(no data)")
        continue

    # Scale bubble size: Cook's D normalised for visibility
    sizes = np.clip(cooks_d[mask] * 1000, 1, 100)

    ax.scatter(
        leverage[mask],
        studentized_resid[mask],
        s=sizes,
        alpha=0.2,
        edgecolors="none",
    )
    ax.axhline(3, color="red", linestyle="--", linewidth=0.8)
    ax.axhline(-3, color="red", linestyle="--", linewidth=0.8)
    ax.axvline(leverage_threshold, color="orange", linestyle=":", linewidth=0.8)
    ax.set_xlabel("Leverage $h_{ii}$")
    if idx == 0:
        ax.set_ylabel("Studentized Residual")
    ax.set_title(f"{meter_label} (meter={meter_code})")

fig.suptitle("Leverage vs Studentized Residuals (bubble size ∝ Cook's D)", fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

---
## Section 6: Residual Plot as Transition Bridge

The residuals-vs-fitted plot below shows **systematic nonlinear structure** — clear evidence of *Lack of Fit*. Rather than forcing a Lack-of-Fit $F$-test (which requires true replicates not present in this continuous-feature dataset), the residual pattern itself constitutes sufficient diagnostic evidence to motivate nonlinear modelling.

We overlay a **LOWESS smoother** (locally weighted scatterplot smoothing) to make the nonlinear trend explicit. If the OLS model were correctly specified, the LOWESS curve would be flat at zero. Any deviation reveals structure the linear model fails to capture.

### Residuals vs Fitted with LOWESS smoother

In [ ]:
import statsmodels.nonparametric.smoothers_lowess as lowess_mod

fig, ax = plt.subplots(figsize=(12, 6))

# Scatter
ax.scatter(fitted, residuals, alpha=0.06, s=3, edgecolors="none", color="#3498db")
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")

# LOWESS on a subsample (LOWESS is O(n²); 10k points is sufficient)
subsample_idx = rng.choice(len(fitted), size=min(10_000, len(fitted)), replace=False)
lowess_result = lowess_mod.lowess(
    residuals.values[subsample_idx],
    fitted.values[subsample_idx],
    frac=0.15,
    return_sorted=True,
)
ax.plot(lowess_result[:, 0], lowess_result[:, 1], color="red", linewidth=2.5, label="LOWESS smoother")

ax.set_xlabel("Fitted values (log-space)", fontsize=12)
ax.set_ylabel("Residuals", fontsize=12)
ax.set_title("Residuals vs Fitted — Evidence of Nonlinear Lack of Fit", fontsize=13)
ax.legend(fontsize=11)
fig.tight_layout()
plt.show()

---
## Summary & Transition

| Assumption | Test | Verdict |
|---|---|---|
| No multicollinearity | VIF, $\kappa(X^\top X)$ | Possible concern — inspect flagged features |
| Homoscedasticity | Breusch–Pagan | **Violated** — variance depends on $X$ |
| Normality of residuals | Shapiro–Wilk, Q-Q plot | **Violated** — heavy tails |
| Linearity (correct specification) | Residuals vs Fitted + LOWESS | **Violated** — systematic nonlinear structure |

Every core assumption of the classical linear model is violated. This is not a failure — it is **diagnostic evidence** that motivates the next steps:

1. **Notebook 03** — Box–Cox transformation of the target + Weighted Least Squares (WLS) to address heteroscedasticity while remaining in the linear framework.
2. **Notebook 04** — Transition to gradient-boosted tree models (LightGBM, XGBoost), which are assumption-free and can capture the nonlinear structure revealed by the LOWESS smoother.

The key portfolio takeaway: we do not abandon OLS because it is "bad" — we abandon it because we have *measured, tested, and visualised* the specific ways in which reality violates its assumptions.